# SNR-Auswertung der Trainingsdaten (ohne Datenveraenderung)

Dieses Notebook analysiert nur den Ist-Zustand deiner Daten (noisy/clean), damit du fair mit anderen Papers vergleichen kannst.
Es veraendert keine Audiodateien und fuehrt kein Training aus.

## 1) Umgebung und Pfade

In [ ]:
import os
import sys
from pathlib import Path

# Portabel: wenn Notebook direkt im Projekt ausgefuehrt wird, nutze CWD.
# Sonst fallback auf den Workspace-Pfad.
cwd = Path.cwd().resolve()
if cwd.name == 'gankan-lwf':
    PROJECT = cwd
elif (cwd / 'gankan-lwf').is_dir():
    PROJECT = (cwd / 'gankan-lwf').resolve()
else:
    PROJECT = Path('/workspaces/MetricGAN-KAN-LwF/gankan-lwf')

ROOT = PROJECT.parent
hparams_yaml = PROJECT / 'results' / 'results' / 'hparams' / 'train_mgk_g4_d4' / 'hyperparams.yaml'

# Option A: Daten-Root setzen (enthaelt train.json/valid.json/test.json)
data_folder_override = None

# Option B: JSON-Dateien direkt setzen (ueberschreibt data_folder)
train_annotation_override = None
valid_annotation_override = None
test_annotation_override = None

# Fuer schnellen Test optional begrenzen, sonst None
max_files_per_split = None

required_subdirs = [
    'clean_trainset_28spk_wav_16k',
    'noisy_trainset_28spk_wav_16k',
    'clean_testset_wav_16k',
    'noisy_testset_wav_16k',
    'trainset_28spk_txt',
    'testset_txt',
]


def is_valid_voicebank_root(path: Path) -> bool:
    return path.is_dir() and all((path / d).exists() for d in required_subdirs)


def autodetect_voicebank_root(project_dir: Path):
    candidates = [
        project_dir / 'data',
        project_dir.parent / 'data',
        project_dir.parent.parent / 'data',
        Path.home() / 'speechbrain' / 'data',
        Path.home() / 'speechbrain' / 'data' / 'noisy-vctk-16k',
    ]
    for c in candidates:
        if is_valid_voicebank_root(c):
            return str(c.resolve())
        nested = c / 'noisy-vctk-16k'
        if is_valid_voicebank_root(nested):
            return str(nested.resolve())
    return None


auto_data_folder = autodetect_voicebank_root(PROJECT)

print('Python:', sys.version.split()[0])
print('Projekt:', PROJECT)
print('hparams:', hparams_yaml)
print('CUDA sichtbar:', os.environ.get('CUDA_VISIBLE_DEVICES', '(nicht gesetzt)'))
print('Auto-detect data_folder:', auto_data_folder)
print('\nKonfig-Hinweis:')
print('- Prioritaet 1: direkte JSON-Overrides')
print('- Prioritaet 2: data_folder_override')
print('- Prioritaet 3: auto-detect wie in run_train.ipynb')
print('- Es werden nur Dateien gelesen, nichts im Datensatz geschrieben.')

In [ ]:
import importlib

required = ['numpy', 'pandas', 'soundfile', 'matplotlib']
missing = [pkg for pkg in required if importlib.util.find_spec(pkg) is None]

if missing:
    print('Fehlende Pakete:', missing)
    print('Bitte installieren: pip install ' + ' '.join(missing))

import numpy as np
import pandas as pd
import soundfile as sf
import matplotlib.pyplot as plt
from IPython.display import display

pesq_available = importlib.util.find_spec('pesq') is not None
stoi_available = importlib.util.find_spec('pystoi') is not None

if pesq_available:
    from pesq import pesq
if stoi_available:
    from pystoi import stoi

print('Imports OK.')
print('PESQ verfuegbar:', pesq_available)
print('STOI verfuegbar:', stoi_available)

## 2) Abhaengigkeiten und Imports

## 3) Konfiguration aus train.yaml lesen

In [ ]:
import json
import re


def parse_key_from_yaml(text, key):
    pat = rf"^{key}:\s*(.+)$"
    for line in text.splitlines():
        m = re.match(pat, line.strip())
        if m:
            return m.group(1).strip()
    return None


def strip_sb_ref(value):
    if value is None:
        return None
    value = value.replace('!ref', '').strip()
    if value.startswith('<') and value.endswith('>'):
        value = value[1:-1]
    return value


def is_placeholder_path(value):
    if value is None:
        return True
    v = str(value).strip()
    return (
        v == ''
        or 'PLACEHOLDER' in v
        or '<data_folder>' in v
        or v == 'data_folder'
    )


def _resolve_existing_dir(path_like, anchors):
    if path_like is None:
        return None
    p = Path(str(path_like)).expanduser()
    candidates = []
    if p.is_absolute():
        candidates.append(p)
    else:
        candidates.append(p)
        for a in anchors:
            candidates.append((a / p).resolve())
    for c in candidates:
        if c.is_dir():
            return str(c.resolve())
    return str(p)


def _resolve_existing_file(path_like, anchors):
    if path_like is None:
        return None
    p = Path(str(path_like)).expanduser()
    candidates = []
    if p.is_absolute():
        candidates.append(p)
    else:
        candidates.append(p)
        for a in anchors:
            candidates.append((a / p).resolve())
    for c in candidates:
        if c.is_file():
            return str(c.resolve())
    return str(p)


def _replace_data_folder_token(path_like, data_folder):
    if path_like is None:
        return None
    p = str(path_like)
    if '{data_root}' in p and data_folder is not None:
        p = p.replace('{data_root}', str(data_folder))
    if '<data_folder>' in p and data_folder is not None:
        p = p.replace('<data_folder>', str(data_folder))
    return p


def load_hparams_paths(yaml_path):
    txt = Path(yaml_path).read_text(encoding='utf-8')
    out = {
        'data_folder': strip_sb_ref(parse_key_from_yaml(txt, 'data_folder')),
        'train_annotation': strip_sb_ref(parse_key_from_yaml(txt, 'train_annotation')),
        'valid_annotation': strip_sb_ref(parse_key_from_yaml(txt, 'valid_annotation')),
        'test_annotation': strip_sb_ref(parse_key_from_yaml(txt, 'test_annotation')),
    }
    return out


cfg = load_hparams_paths(hparams_yaml)
anchors = [PROJECT, PROJECT.parent, Path.cwd().resolve()]

# Prioritaet: expliziter Override > Auto-detect > YAML
if data_folder_override:
    cfg['data_folder'] = str(Path(data_folder_override).expanduser())
elif auto_data_folder:
    cfg['data_folder'] = auto_data_folder

# Wenn data_folder noch Placeholder ist, auf None setzen (sauberere Fehlermeldung)
if is_placeholder_path(cfg.get('data_folder')):
    cfg['data_folder'] = None
else:
    cfg['data_folder'] = _resolve_existing_dir(cfg['data_folder'], anchors)

# Direkte JSON-Overrides haben Prioritaet
if train_annotation_override:
    cfg['train_annotation'] = str(Path(train_annotation_override).expanduser())
if valid_annotation_override:
    cfg['valid_annotation'] = str(Path(valid_annotation_override).expanduser())
if test_annotation_override:
    cfg['test_annotation'] = str(Path(test_annotation_override).expanduser())

# Relative/templated Annotation-Pfade robust aufloesen
for split_key in ['train_annotation', 'valid_annotation', 'test_annotation']:
    raw = cfg.get(split_key)
    raw = _replace_data_folder_token(raw, cfg.get('data_folder'))
    cfg[split_key] = _resolve_existing_file(raw, anchors)

print('Konfiguration:')
for k, v in cfg.items():
    print(f'- {k}: {v}')

In [ ]:
def replace_data_root(path_like, data_root):
    if path_like is None:
        return None
    p = str(path_like)
    if '{data_root}' in p:
        if data_root is None:
            return p.replace('{data_root}', '<BITTE_DATA_FOLDER_SETZEN>')
        return str(Path(p.replace('{data_root}', str(data_root))))
    return str(Path(p))


def load_annotation_json(json_path):
    p = Path(json_path)
    if not p.exists():
        raise FileNotFoundError(f'Annotation nicht gefunden: {p}')
    with p.open('r', encoding='utf-8') as f:
        data = json.load(f)

    rows = []
    for utt_id, item in data.items():
        rows.append({
            'id': utt_id,
            'noisy_wav': item.get('noisy_wav'),
            'clean_wav': item.get('clean_wav'),
        })
    return pd.DataFrame(rows)


def read_audio_mono(path):
    wav, sr = sf.read(path)
    if wav.ndim > 1:
        wav = wav.mean(axis=1)
    return wav.astype(np.float64), int(sr)


def snr_db(clean, noisy, eps=1e-10):
    noise = noisy - clean
    p_sig = float(np.mean(clean ** 2))
    p_noise = float(np.mean(noise ** 2))
    return 10.0 * np.log10((p_sig + eps) / (p_noise + eps))

## 4) Annotationen laden (nur Pfadinspektion, keine Datenaenderung)

## 5) Noisy-Baseline auswerten (SNR, optional PESQ/STOI)

In [ ]:
def resolve_annotation_path(path_template, data_folder):
    if path_template is None:
        return None
    path_template = str(path_template)

    # Bereits absolut/relativ als direkter Dateipfad
    p = Path(path_template)
    if p.exists():
        return str(p)

    if '{data_root}' in path_template:
        if data_folder is None:
            return path_template.replace('{data_root}', '<BITTE_DATA_FOLDER_SETZEN>')
        return path_template.replace('{data_root}', str(data_folder))

    if '<data_folder>' in path_template:
        if data_folder is None:
            return path_template.replace('<data_folder>', '<BITTE_DATA_FOLDER_SETZEN>')
        return path_template.replace('<data_folder>', str(data_folder))

    return path_template


ann_paths = {
    'train': resolve_annotation_path(cfg['train_annotation'], cfg['data_folder']),
    'valid': resolve_annotation_path(cfg['valid_annotation'], cfg['data_folder']),
    'test': resolve_annotation_path(cfg['test_annotation'], cfg['data_folder']),
}

splits = {}
missing = []
for split, ann_path in ann_paths.items():
    try:
        if ann_path is None:
            raise FileNotFoundError('Kein Pfad gesetzt')
        df = load_annotation_json(ann_path)
        if max_files_per_split is not None:
            df = df.head(int(max_files_per_split)).copy()
        df['noisy_path'] = df['noisy_wav'].apply(lambda x: replace_data_root(x, cfg['data_folder']))
        df['clean_path'] = df['clean_wav'].apply(lambda x: replace_data_root(x, cfg['data_folder']))
        splits[split] = df
        print(f'{split}: {len(df)} Paare | Annotation: {ann_path}')
    except Exception as e:
        missing.append((split, ann_path, str(e)))
        print(f'{split}: konnte nicht geladen werden ({e})')

if missing:
    print('\nFix: Setze oben in Zelle 3 entweder:')
    print('1) data_folder_override = "/dein/pfad/mit/train.json"')
    print('oder')
    print('2) train_annotation_override / valid_annotation_override / test_annotation_override direkt.')
    print('\nAktuell aufgeloeste Pfade:')
    for split, path, err in missing:
        print(f'- {split}: {path}')

for split, df in splits.items():
    display(df.head(3))

In [ ]:
try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kwargs):
        return x


def eval_split(df, split_name):
    rows = []
    skipped_missing = 0
    skipped_sr = 0
    skipped_len = 0
    skipped_short = 0

    for _, r in tqdm(df.iterrows(), total=len(df), desc=f'{split_name} eval'):
        noisy_path = Path(r['noisy_path'])
        clean_path = Path(r['clean_path'])

        if not noisy_path.exists() or not clean_path.exists():
            skipped_missing += 1
            continue

        try:
            noisy, sr_n = read_audio_mono(noisy_path)
            clean, sr_c = read_audio_mono(clean_path)
        except Exception:
            skipped_missing += 1
            continue

        # Keine Datenanpassung: unterschiedliche SR oder Laengen werden uebersprungen.
        if sr_n != sr_c:
            skipped_sr += 1
            continue
        if len(noisy) != len(clean):
            skipped_len += 1
            continue
        if len(noisy) < 16:
            skipped_short += 1
            continue

        row = {
            'id': r['id'],
            'split': split_name,
            'sr': sr_n,
            'n_samples': len(noisy),
            'snr_db': snr_db(clean, noisy),
            'noisy_path': str(noisy_path),
            'clean_path': str(clean_path),
        }

        if pesq_available and sr_n == 16000:
            try:
                row['pesq_noisy'] = pesq(16000, clean, noisy, 'wb')
            except Exception:
                row['pesq_noisy'] = np.nan
        else:
            row['pesq_noisy'] = np.nan

        if stoi_available:
            try:
                row['stoi_noisy'] = stoi(clean, noisy, sr_n, extended=False)
            except Exception:
                row['stoi_noisy'] = np.nan
        else:
            row['stoi_noisy'] = np.nan

        rows.append(row)

    out = pd.DataFrame(rows)
    skip_info = {
        'split': split_name,
        'input_pairs': len(df),
        'used_pairs': len(out),
        'skipped_missing_or_read_error': skipped_missing,
        'skipped_sr_mismatch': skipped_sr,
        'skipped_length_mismatch': skipped_len,
        'skipped_too_short': skipped_short,
    }
    return out, skip_info


all_rows = []
skip_rows = []
for split, df in splits.items():
    out, info = eval_split(df, split)
    all_rows.append(out)
    skip_rows.append(info)

metrics_df = pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame()
skip_df = pd.DataFrame(skip_rows)

display(skip_df)

if metrics_df.empty:
    print('Keine auswertbaren Paare gefunden.')
else:
    summary = metrics_df.groupby('split')[['snr_db', 'pesq_noisy', 'stoi_noisy']].agg(['count', 'mean', 'median', 'std', 'min', 'max']).round(3)
    display(summary)

    bins = [-np.inf, -5, 0, 5, 10, 15, 20, np.inf]
    labels = ['<-5', '-5..0', '0..5', '5..10', '10..15', '15..20', '>20']
    metrics_df['snr_bin_db'] = pd.cut(metrics_df['snr_db'], bins=bins, labels=labels)

    bin_table = (
        metrics_df.groupby(['split', 'snr_bin_db'])
        .agg(
            n=('id', 'count'),
            snr_mean=('snr_db', 'mean'),
            pesq_mean=('pesq_noisy', 'mean'),
            stoi_mean=('stoi_noisy', 'mean'),
        )
        .reset_index()
    )
    display(bin_table)

    for split in sorted(metrics_df['split'].unique()):
        sub = metrics_df[metrics_df['split'] == split]
        print(f'\n{split}: niedrigste SNR-Beispiele')
        display(sub.nsmallest(10, 'snr_db')[['id', 'snr_db', 'pesq_noisy', 'stoi_noisy', 'noisy_path']])

In [ ]:
if metrics_df.empty:
    print('Keine Metriken vorhanden.')
else:
    splits_sorted = sorted(metrics_df['split'].unique())
    fig, axes = plt.subplots(len(splits_sorted), 1, figsize=(10, 3 * len(splits_sorted)), squeeze=False)
    for i, split in enumerate(splits_sorted):
        ax = axes[i, 0]
        sub = metrics_df[metrics_df['split'] == split]
        ax.hist(sub['snr_db'].dropna().values, bins=40, alpha=0.8, color='#1f77b4')
        ax.set_title(f'SNR-Verteilung ({split})')
        ax.set_xlabel('SNR [dB]')
        ax.set_ylabel('Anzahl')
    plt.tight_layout()
    plt.show()

## 6) Table-1-kompatibler Vergleich: Noisy vs MetricGAN vs MetricGAN-LwF

In [ ]:
# Vergleichsbasis und Ablationen konfigurieren
ablation_root_env = os.environ.get('METRICGAN_ABLATION_ROOT')
default_ablation_root = Path.cwd() / 'results' / 'ablations_core'
ablation_root = Path(ablation_root_env).expanduser() if ablation_root_env else default_ablation_root

baseline_run_name = 'baseline_lwf_off'

# None => alle Runs unter ablation_root (ausser baseline_run_name)
compare_run_names = None

# Optional manuell, z. B.:
# compare_run_names = ['baseline_lambda0', 'fixed_l1_n10', 'fixed_l0p25_n50']

# Vergleich auf diesen nominalen SNR-Stufen wie im Paper
target_snr_levels = [-12, -6, 0, 6, 12]
snr_tolerance_db = 1.0

# Normalerweise test verwenden fuer Papervergleich
eval_split_for_table = 'test'

# Mindestanzahl pro SNR-Stufe; darunter Warnung
min_samples_per_level = 20


def _list_ablation_runs(root_dir: Path):
    if not root_dir.is_dir():
        return []
    return sorted([p.name for p in root_dir.iterdir() if p.is_dir()])


def _resolve_run_enhanced_dir(root_dir: Path, run_name: str):
    run_dir = root_dir / run_name
    candidates = [run_dir / 'enhanced_wavs']

    # Unterordner wie seed (z. B. 4234/enhanced_wavs) mit aufnehmen
    if run_dir.is_dir():
        for child in sorted(run_dir.iterdir()):
            if child.is_dir():
                candidates.append(child / 'enhanced_wavs')

    for c in candidates:
        if c.is_dir():
            return str(c.resolve())
    return None


available_runs = _list_ablation_runs(ablation_root)
if compare_run_names is None:
    selected_runs = [r for r in available_runs if r != baseline_run_name]
else:
    selected_runs = [r for r in compare_run_names if r != baseline_run_name]

baseline_enhanced_dir = _resolve_run_enhanced_dir(ablation_root, baseline_run_name)
run_to_enhanced_dir = {r: _resolve_run_enhanced_dir(ablation_root, r) for r in selected_runs}

print('Konfiguration gesetzt.')
print('Ablation-Root:', ablation_root)
print('Baseline-Run:', baseline_run_name)
print('Baseline enhanced_dir:', baseline_enhanced_dir)
print('Anzahl Vergleichs-Runs:', len(selected_runs))
print('Split:', eval_split_for_table)
print('SNR-Ziele:', target_snr_levels)
print('Toleranz ± dB:', snr_tolerance_db)

missing_runs = [r for r, p in run_to_enhanced_dir.items() if p is None]
if missing_runs:
    print('Warnung: Fuer folgende Runs kein enhanced_wavs gefunden:')
    print(missing_runs[:20])

print('Beispiel Run-Pfade (erste 10):')
for r in selected_runs[:10]:
    print(f'- {r}: {run_to_enhanced_dir.get(r)}')

In [ ]:
def _parse_epoch_from_stem(stem):
    if '@' not in stem:
        return -1
    tail = stem.rsplit('@', 1)[1]
    return int(tail) if tail.isdigit() else -1


def build_enhanced_index(enhanced_dir):
    wav_paths = sorted(Path(enhanced_dir).rglob('*.wav'))
    by_exact = {}
    by_base = {}
    for p in wav_paths:
        stem = p.stem
        by_exact.setdefault(stem, []).append(p)
        base = stem.split('@', 1)[0]
        by_base.setdefault(base, []).append(p)
    return wav_paths, by_exact, by_base


def _choose_best_match(paths):
    if not paths:
        return None
    return sorted(paths, key=lambda x: (_parse_epoch_from_stem(x.stem), x.name))[-1]


def resolve_enhanced_path(utt_id, by_exact, by_base):
    stem = Path(str(utt_id).strip()).stem
    if stem in by_exact:
        return _choose_best_match(by_exact[stem])
    if stem in by_base:
        return _choose_best_match(by_base[stem])
    return None


def eval_enhanced_for_ids(base_df, enhanced_dir, model_name):
    if enhanced_dir is None:
        print(f'{model_name}: enhanced_dir ist None, Abbruch fuer dieses Modell.')
        return pd.DataFrame(columns=['id', 'split', 'snr_level', 'pesq', 'stoi'])

    enhanced_dir = Path(enhanced_dir)
    if not enhanced_dir.is_dir():
        print(f'{model_name}: enhanced_dir existiert nicht: {enhanced_dir}')
        return pd.DataFrame(columns=['id', 'split', 'snr_level', 'pesq', 'stoi'])

    wav_paths, by_exact, by_base = build_enhanced_index(enhanced_dir)
    print(f'{model_name}: {len(wav_paths)} WAV-Dateien rekursiv gefunden unter {enhanced_dir}')
    if len(wav_paths) == 0:
        return pd.DataFrame(columns=['id', 'split', 'snr_level', 'pesq', 'stoi'])

    requested_ids = set(base_df['id'].astype(str).map(lambda x: Path(x).stem))
    available_ids = set(by_base.keys())
    overlap = requested_ids & available_ids
    print(
        f"{model_name}: ID-Overlap requested={len(requested_ids)} | available={len(available_ids)} | overlap={len(overlap)}"
    )
    if len(overlap) == 0:
        print(f"{model_name}: Beispiel available IDs: {sorted(list(available_ids))[:5]}")

    rows = []
    miss_clean = 0
    miss_enh = 0
    read_err = 0
    sr_mismatch = 0
    len_mismatch = 0
    too_short = 0

    for _, r in base_df.iterrows():
        clean_path = Path(r['clean_path'])
        enh_path = resolve_enhanced_path(r['id'], by_exact, by_base)

        if not clean_path.exists():
            miss_clean += 1
            continue
        if enh_path is None or not enh_path.exists():
            miss_enh += 1
            continue

        try:
            enh, sr_e = read_audio_mono(enh_path)
            clean, sr_c = read_audio_mono(clean_path)
        except Exception:
            read_err += 1
            continue

        if sr_e != sr_c:
            sr_mismatch += 1
            continue
        if len(enh) != len(clean):
            len_mismatch += 1
            continue
        if len(enh) < 16:
            too_short += 1
            continue

        row = {
            'id': r['id'],
            'split': r['split'],
            'snr_level': r['snr_level'],
            'pesq': np.nan,
            'stoi': np.nan,
        }

        if pesq_available and sr_e == 16000:
            try:
                row['pesq'] = pesq(16000, clean, enh, 'wb')
            except Exception:
                row['pesq'] = np.nan

        if stoi_available:
            try:
                row['stoi'] = stoi(clean, enh, sr_e, extended=False)
            except Exception:
                row['stoi'] = np.nan

        rows.append(row)

    out = pd.DataFrame(rows)
    print(f'{model_name}: {len(out)} gueltige Enhanced-Paare ausgewertet.')
    print(
        f"{model_name} Skip-Stats | clean fehlt: {miss_clean} | enhanced fehlt: {miss_enh} | "
        f"read error: {read_err} | SR mismatch: {sr_mismatch} | Len mismatch: {len_mismatch} | zu kurz: {too_short}"
    )
    return out


def summarize_levels(compare_df, level_col='snr_level'):
    agg = compare_df.groupby(level_col).agg(
        n=('id', 'count'),
        noisy_pesq=('pesq_noisy', 'mean'),
        noisy_stoi=('stoi_noisy', 'mean'),
        baseline_pesq=('pesq_baseline', 'mean'),
        baseline_stoi=('stoi_baseline', 'mean'),
        run_pesq=('pesq_run', 'mean'),
        run_stoi=('stoi_run', 'mean'),
    ).reset_index()
    return agg


if metrics_df.empty:
    print('Keine Baseline-Metriken vorhanden. Fuehre zuerst die Baseline-Zellen aus.')
else:
    base_split = metrics_df[metrics_df['split'] == eval_split_for_table].copy()

    if base_split.empty:
        print('Gewaehlter Split hat keine Daten:', eval_split_for_table)
    else:
        level_frames = []
        for lvl in target_snr_levels:
            sub = base_split[(base_split['snr_db'] >= (lvl - snr_tolerance_db)) & (base_split['snr_db'] <= (lvl + snr_tolerance_db))].copy()
            sub['snr_level'] = lvl
            level_frames.append(sub)

        level_df = pd.concat(level_frames, ignore_index=True) if level_frames else pd.DataFrame()

        if level_df.empty:
            print('Keine Samples in den SNR-Fenstern gefunden. Erhoehe ggf. snr_tolerance_db.')
        else:
            counts = level_df.groupby('snr_level')['id'].count().reset_index(name='n')
            print('Baseline-Samples pro SNR-Level:')
            display(counts)

            low_counts = counts[counts['n'] < min_samples_per_level]
            if not low_counts.empty:
                print('Warnung: Kleine Bin-Groessen gefunden:')
                display(low_counts)

            base_eval = level_df[['id', 'split', 'clean_path', 'snr_level']].copy()

            base_model_name = f'Baseline ({baseline_run_name})'
            baseline_df = eval_enhanced_for_ids(base_eval, baseline_enhanced_dir, base_model_name)
            if baseline_df.empty:
                print('Keine gueltigen Enhanced-Paare fuer die Baseline gefunden. Abbruch.')
            else:
                all_run_tables = []
                keys = ['id', 'split', 'snr_level']

                for run_name in selected_runs:
                    run_enhanced_dir = run_to_enhanced_dir.get(run_name)
                    run_model_name = f'Run ({run_name})'
                    run_df = eval_enhanced_for_ids(base_eval, run_enhanced_dir, run_model_name)
                    if run_df.empty:
                        continue

                    common_ids = baseline_df[keys].drop_duplicates().merge(
                        run_df[keys].drop_duplicates(),
                        on=keys,
                        how='inner',
                    )
                    if common_ids.empty:
                        print(f'{run_name}: Keine gemeinsame ID-Menge mit Baseline gefunden.')
                        continue

                    compare = level_df[['id', 'split', 'snr_db', 'snr_level', 'pesq_noisy', 'stoi_noisy']].copy()
                    compare = compare.merge(common_ids, on=keys, how='inner')
                    compare = compare.merge(
                        baseline_df.rename(columns={'pesq': 'pesq_baseline', 'stoi': 'stoi_baseline'}),
                        on=keys,
                        how='left',
                    )
                    compare = compare.merge(
                        run_df.rename(columns={'pesq': 'pesq_run', 'stoi': 'stoi_run'}),
                        on=keys,
                        how='left',
                    )

                    common_counts = compare.groupby('snr_level')['id'].count().reset_index(name='n_common')
                    print(f'{run_name}: Gemeinsame ID-Menge pro SNR-Level:')
                    display(common_counts)

                    run_table = summarize_levels(compare).sort_values('snr_level').reset_index(drop=True)
                    avg_row = {
                        'snr_level': 'Avg.',
                        'n': int(run_table['n'].sum()),
                        'noisy_pesq': run_table['noisy_pesq'].mean(),
                        'noisy_stoi': run_table['noisy_stoi'].mean(),
                        'baseline_pesq': run_table['baseline_pesq'].mean(),
                        'baseline_stoi': run_table['baseline_stoi'].mean(),
                        'run_pesq': run_table['run_pesq'].mean(),
                        'run_stoi': run_table['run_stoi'].mean(),
                    }
                    run_table = pd.concat([run_table, pd.DataFrame([avg_row])], ignore_index=True)
                    run_table.insert(0, 'run_name', run_name)

                    display(run_table.round(3))
                    all_run_tables.append(run_table)

                if all_run_tables:
                    summary_all = pd.concat(all_run_tables, ignore_index=True)
                    print('Kompakte Uebersicht ueber alle verglichenen Runs (inkl. Avg.-Zeilen):')
                    display(summary_all[summary_all['snr_level'] == 'Avg.'].round(3))
                    print('Hinweis: Jeder Vergleich basiert auf exakt der gemeinsamen ID-Menge pro SNR-Stufe.')
                else:
                    print('Kein Run mit gueltiger gemeinsamer ID-Menge zur Baseline gefunden.')

In [ ]:
# 7) Gesamt-Uebersicht: PESQ ueber alle Runs und SNR-Stufen
if 'summary_all' not in globals() or summary_all is None or len(summary_all) == 0:
    print('Keine summary_all-Daten vorhanden. Fuehre zuerst Zelle 16 aus.')
else:
    s = summary_all.copy()
    s = s[s['snr_level'] != 'Avg.'].copy()

    # Delta-Metriken fuer schnelleren Vergleich
    s['delta_run_vs_baseline_pesq'] = s['run_pesq'] - s['baseline_pesq']
    s['delta_run_vs_noisy_pesq'] = s['run_pesq'] - s['noisy_pesq']
    s['delta_baseline_vs_noisy_pesq'] = s['baseline_pesq'] - s['noisy_pesq']

    # Sortierhilfe fuer SNR-Level
    s['snr_level_sort'] = pd.to_numeric(s['snr_level'], errors='coerce')
    s = s.sort_values(['run_name', 'snr_level_sort']).drop(columns=['snr_level_sort'])

    print('Gesamt-Uebersicht PESQ pro Run und SNR-Level:')
    display(
        s[[
            'run_name', 'snr_level', 'n',
            'noisy_pesq', 'baseline_pesq', 'run_pesq',
            'delta_baseline_vs_noisy_pesq',
            'delta_run_vs_noisy_pesq',
            'delta_run_vs_baseline_pesq',
        ]].round(3)
    )

    print('Pivot: run_pesq je Run x SNR-Level')
    run_pesq_pivot = s.pivot_table(
        index='run_name',
        columns='snr_level',
        values='run_pesq',
        aggfunc='mean',
    )
    display(run_pesq_pivot.round(3))

    print('Pivot: Delta(run - baseline) PESQ je Run x SNR-Level')
    delta_pivot = s.pivot_table(
        index='run_name',
        columns='snr_level',
        values='delta_run_vs_baseline_pesq',
        aggfunc='mean',
    )
    display(delta_pivot.round(3))

    print('Run-Ranking nach mittlerem Delta(run - baseline) ueber SNR-Level:')
    ranking = (
        s.groupby('run_name', as_index=False)['delta_run_vs_baseline_pesq']
        .mean()
        .sort_values('delta_run_vs_baseline_pesq', ascending=False)
    )
    display(ranking.round(3))